# Triplet Metric Learning + SVM

Standard triplet-loss counterpart to `02` (quadruplet). For every anchor, mines one hard positive (same class) and one semi-hard/hard negative (different class), and trains an embedding network so that `d(anchor, positive) + margin < d(anchor, negative)`. The learned embeddings are then classified with an SVM, using the exact same evaluation protocol as the quadruplet notebook so the two are directly comparable.

**In this project, the triplet variant outperformed the quadruplet variant** on the held-out test set. Cleaned, importable version of this pipeline: `src/models/triplet_net.py`.

## 1. Setup

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.manifold import TSNE
import umap
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## 2. Load extracted deep features
`features.npy` / `labels.npy` are produced by `src/features/feature_extraction.py` (DenseNet121 + ResNet50 concatenated features).

In [ ]:
X = np.load("features.npy")
y = np.load("labels.npy")
print("Feature shape:", X.shape)
print("Label shape:", y.shape)


## 3. Class imbalance check & train/val split

In [ ]:
# check class imbalance
np.bincount(y)


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


## 4. Embedding-space visualization helpers (t-SNE, UMAP, class-similarity heatmap)

In [ ]:
def plot_tsne(X, y, title):
    if len(X) > 3000:
        idx = np.random.choice(len(X), 3000, replace=False)
        X = X[idx]
        y = y[idx]

    tsne = TSNE(n_components=2, perplexity=40, learning_rate=200, init="pca", random_state=42)
    X_embedded = tsne.fit_transform(X)

    plt.figure(figsize=(6,6))
    plt.scatter(X_embedded[:,0], X_embedded[:,1], c=y, cmap="tab10", s=8)
    plt.title(title)
    plt.show()


def plot_umap(X, y, title):
    reducer = umap.UMAP()
    X_embedded = reducer.fit_transform(X)

    plt.figure(figsize=(6,6))
    plt.scatter(X_embedded[:,0], X_embedded[:,1], c=y, cmap="tab10")
    plt.title(title)
    plt.show()


def class_similarity(X, y):
    classes = np.unique(y)
    means = [X[y==c].mean(axis=0) for c in classes]
    means = np.stack(means)
    sim = np.corrcoef(means)

    plt.figure(figsize=(6,5))
    sns.heatmap(sim, annot=True, cmap="coolwarm")
    plt.title("Class Similarity Matrix")
    plt.show()


## 5. Baseline visualization — before any metric learning

In [ ]:
plot_tsne(X_train, y_train, "t-SNE BEFORE training")
plot_umap(X_train, y_train, "UMAP BEFORE training")
class_similarity(X_train, y_train)


## 6. Dataset / DataLoader

In [ ]:
class FeatureDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X).float()
        self.y = torch.tensor(y).long()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_dataset = FeatureDataset(X_train, y_train)
val_dataset = FeatureDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)


## 7. Triplet mining
For every anchor: hardest same-class positive, and a semi-hard negative just beyond the hardest-positive distance (falling back to the closest negative if none qualifies). Same mining strategy as the quadruplet notebook, minus the second negative — see `src/models/triplet_net.py::mine_triplets` for the cleaned, documented version.

In [ ]:
def mine_triplets(embeddings, labels, margin=1.0):
    dist = torch.cdist(embeddings, embeddings)

    anchors, positives, negatives = [], [], []

    for i in range(len(labels)):
        anchor_label = labels[i]

        pos_mask = labels == anchor_label
        neg_mask = labels != anchor_label

        pos_indices = torch.where(pos_mask)[0]
        neg_indices = torch.where(neg_mask)[0]
        pos_indices = pos_indices[pos_indices != i]

        if len(pos_indices) == 0 or len(neg_indices) == 0:
            continue

        pos_dist = dist[i][pos_indices]
        neg_dist = dist[i][neg_indices]

        hard_pos = pos_indices[torch.argmax(pos_dist)]
        pos_max = pos_dist.max()

        semi_hard_mask = (neg_dist > pos_max) & (neg_dist < pos_max + margin)
        semi_hard = neg_indices[semi_hard_mask]

        if len(semi_hard) > 0:
            hard_neg = semi_hard[torch.randint(0, len(semi_hard), (1,))]
        else:
            hard_neg = neg_indices[torch.argmin(neg_dist)]

        anchors.append(i)
        positives.append(hard_pos.item())
        negatives.append(hard_neg.item())

    return anchors, positives, negatives


## 8. Embedding network (small MLP head on top of the frozen deep features)

In [ ]:
class EmbeddingNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),

            nn.Linear(256, 128)
        )

    def forward(self, x):
        return self.net(x)


model = EmbeddingNet(X.shape[1]).to(device)


## 9. Triplet loss

In [ ]:
def triplet_loss(a, p, n, margin=1.0):
    d_ap = F.pairwise_distance(a, p)
    d_an = F.pairwise_distance(a, n)
    return F.relu(d_ap - d_an + margin).mean()


optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)


## 10. Training loop (100 epochs)

In [ ]:
epochs = 100
margin = 1.0

for epoch in range(epochs):

    model.train()

    total_loss = 0
    total_dap = 0
    total_dan = 0

    num_batches = 0
    dist_batches = 0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

    for X_batch, y_batch in progress_bar:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        embeddings = model(X_batch)

        a, p, n = mine_triplets(embeddings, y_batch, margin=margin)

        if len(a) == 0:
            continue

        anchor = embeddings[a]
        positive = embeddings[p]
        negative = embeddings[n]

        d_ap = F.pairwise_distance(anchor, positive)
        d_an = F.pairwise_distance(anchor, negative)

        total_dap += d_ap.mean().item()
        total_dan += d_an.mean().item()
        dist_batches += 1

        loss = triplet_loss(anchor, positive, negative, margin=margin)
        loss += 0.001 * torch.mean(torch.norm(embeddings, dim=1))

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

        progress_bar.set_postfix(loss=loss.item())

    avg_loss = total_loss / num_batches
    avg_dap = total_dap / dist_batches
    avg_dan = total_dan / dist_batches

    print(f"\nEpoch {epoch+1}")
    print(f"Train Loss: {avg_loss:.4f}")
    print(f"Mean d(a,p): {avg_dap:.4f}")
    print(f"Mean d(a,n): {avg_dan:.4f}")


## 11. Embedding-space visualization — after metric learning
Compare against step 5 to see how much tighter/more separable the classes become.

In [ ]:
model.eval()

with torch.no_grad():
    embeddings = model(torch.tensor(X).float().to(device)).cpu().numpy()

plot_tsne(embeddings, y, "t-SNE AFTER training")
plot_umap(embeddings, y, "UMAP AFTER training")
class_similarity(embeddings, y)


## 12. Extract embeddings for the train/val split

In [ ]:
model.eval()

def extract_embeddings(model, loader):
    all_embeddings = []
    all_labels = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            embeddings = model(X_batch)
            all_embeddings.append(embeddings.cpu())
            all_labels.append(y_batch)

    embeddings = torch.cat(all_embeddings).numpy()
    labels = torch.cat(all_labels).numpy()

    return embeddings, labels


X_train_emb, y_train_emb = extract_embeddings(model, train_loader)
X_val_emb, y_val_emb = extract_embeddings(model, val_loader)


## 13. Train an SVM on the learned embeddings

In [ ]:
from sklearn.svm import SVC

svm = SVC(
    kernel="rbf",
    C=10,
    gamma="scale",
    class_weight="balanced"
)

svm.fit(X_train_emb, y_train_emb)


In [ ]:
y_pred = svm.predict(X_val_emb)


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_val_emb, y_pred))


In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_val_emb, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()


In [ ]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score

print("Accuracy:", accuracy_score(y_val_emb, y_pred))
print("Balanced accuracy:", balanced_accuracy_score(y_val_emb, y_pred))


In [ ]:
from sklearn.metrics import f1_score

print("Macro F1:", f1_score(y_val_emb, y_pred, average="macro"))
print("Weighted F1:", f1_score(y_val_emb, y_pred, average="weighted"))


## 14. Final evaluation on the held-out test set

In [ ]:
X_test = np.load("test_features.npy")
y_test = np.load("test_labels.npy")

print("Test feature shape:", X_test.shape)
print("Test label shape:", y_test.shape)
print("Unique test labels:", np.unique(y_test))

test_loader = DataLoader(FeatureDataset(X_test, y_test), batch_size=64, shuffle=False)

model.eval()
all_embeddings = []
all_labels = []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        emb = model(xb)
        all_embeddings.append(emb.cpu())
        all_labels.append(yb)

X_test_emb = torch.cat(all_embeddings).numpy()
y_test_emb = torch.cat(all_labels).numpy()

y_pred_test = svm.predict(X_test_emb)

print("\nClassification Report (TEST):")
print(classification_report(y_test_emb, y_pred_test, digits=4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_emb, y_pred_test))
